# Inside Gemma: a role-play and safety explorer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/unkarelian/gemma-roleplay-interpretability/blob/main/colab/gemma_roleplay_explorer.ipynb)

This notebook runs **Gemma 3 4B-it** (Google's small open chat model) and lets you look inside it while it answers. You type a conversation, the model replies, and an interactive page shows what happened at every token and in every one of its 34 layers:

- **Axes**: directions in the model's internal state, scaled so that 0 and 1 stand for two kinds of prompt. Examples: *plain assistant vs role-play character*, *kind vs hostile*.
- **Probes**: readouts at a single layer. *Crisis* (layer 17) estimates whether the user is in real danger. *Refusal* (layer 14) measures how far the state has moved toward "harmful request".
- **Features**: the units of the [Gemma Scope 2](https://huggingface.co/google/gemma-scope-2-4b-it) transcoders. Each one tends to fire on one kind of text. They come with labels and example texts.
- **Logit lens, attention, step-by-step replay, a watch list of interesting features, side-by-side comparison, and steering** (nudging the model along a direction while it writes).

It comes from a research project on how Gemma handles role-play. The project asked three things: does a character still notice when the *user* is in real danger, how do refusals work in character, and what does "being a character" look like inside the model? The page includes a short list of what the project found.

**Content note.** Some presets are harmful requests (which the model refuses) or messages from a user in crisis, because that is what the project studies. If anything like that is happening to you, please contact your local emergency number or a crisis line.

### What you need
1. **A GPU runtime.** In Colab: *Runtime → Change runtime type → T4 GPU* (free). An L4 or A100 (paid) is faster.
2. **A Hugging Face account with access to Gemma.** Gemma's weights are free but gated:
   1. Sign in at [huggingface.co](https://huggingface.co), open [google/gemma-3-4b-it](https://huggingface.co/google/gemma-3-4b-it) and accept Google's licence. Approval is usually immediate.
   2. Create an access token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens). The *Read* type is enough.
   3. In Colab, click the **key icon** in the left sidebar (*Secrets*), add a secret named `HF_TOKEN` with your token as the value, and switch on *Notebook access*. If you skip this, the notebook asks for the token in a hidden prompt instead. The token is never printed or saved in the notebook.
3. **Run the cells from top to bottom** (*Runtime → Run all* works too). The first start takes about 5–10 minutes. It downloads about 8.6 GB of model weights and 5.7 GB of transcoders.

### Speed and memory (estimates; see the README)
| GPU | GPU memory | One run (80 reply tokens) |
|---|---|---|
| T4 (free, 15 GB) | about 7.5 GB loaded, peaks of roughly 9–11 GB during a run; about 3–4 GB of extra CPU RAM for the transcoders | roughly 15–40 s |
| L4 (24 GB) / A100 (40 GB) | about 11–14 GB (transcoders on the GPU) | roughly 3–10 s |

The model stays in **bfloat16**. The T4 has no native bfloat16 support, so PyTorch emulates it: slower, but the numbers stay right. float16 is not used, because Gemma 3's activations are known to overflow it (they turn into infinity), and that would corrupt exactly the numbers this tool shows. On a T4 the transcoders live in CPU memory and visit the GPU one layer at a time, so everything fits.

### Please read the labels with care
Feature labels are **best guesses written by an LLM** from Gemma Scope's example texts and the project's own prompts. They are hints, not proofs. The axes and probes were **fit on the project's own prompts**, so they describe those kinds of conversation best.

### A short glossary
- **Token**: a piece of text the model reads or writes, often a word or part of one.
- **Layer**: the model processes each token in 34 steps (layers). Early layers mostly handle words and spelling. Middle layers handle meaning. Late layers handle what to say next.
- **Residual stream**: the running internal state for each token. Every layer reads it and adds to it. It is a list of 2,560 numbers per token.
- **Direction / axis**: a fixed pattern in those 2,560 numbers. Projecting the state onto it gives one number, rescaled here so that 0 and 1 are the averages of two groups of example prompts.
- **Probe**: the same idea, read at one chosen layer only (crisis at layer 17, refusal at layer 14).
- **Transcoder feature**: a learned unit (16,384 per layer) that turns the state into something more readable. Each feature tends to fire on one kind of text. Labels are guesses.
- **Logit lens**: what the model would say next if it stopped at this layer.
- **Attention**: which earlier tokens a token is looking at.
- **Steering**: adding a direction to the state while the model writes, to see how its behaviour changes.
- **Last prompt token**: the position just before the reply starts. This is where the model settles what to say, so most axes are easiest to read there.

## 1. Settings

In [ ]:
#@title Settings { display-mode: "form" }
# Where the app files come from: the public GitHub repo that holds the colab/ folder.
REPO_URL = "https://github.com/unkarelian/gemma-roleplay-interpretability"   #@param {type:"string"}
REPO_BRANCH = "main"                                  #@param {type:"string"}
REPO_SUBDIR = "colab"                                 #@param {type:"string"}
# If a folder named explorer/ (with app.py inside) sits next to this notebook or in /content, it is used instead.

SAVE_TO_DRIVE = False            #@param {type:"boolean"}
DRIVE_FOLDER = "gemma_explorer"  #@param {type:"string"}
# Keep runs and notes in your Google Drive so they survive the session. Off: they live in /content and vanish when the runtime ends.

PRELOAD_TRANSCODERS = True       #@param {type:"boolean"}
# True: fetch all 34 transcoder layers now (about 5.7 GB, 1-3 min). False: each layer is fetched the first time you open it.
MY_CONCEPT_GROUPS = ""           #@param {type:"string"}
# Optional: path to a JSON file with your own prompt groups, to fit extra directions at every layer (see section 10).
DTYPE = "bfloat16"               #@param ["bfloat16", "float16"]
# Keep bfloat16. float16 is only here so you can check the overflow problem for yourself (runs then warn about inf/NaN).
PORT = 8000                      #@param {type:"integer"}

import sys, os
IN_COLAB = "google.colab" in sys.modules
BASE = "/content" if IN_COLAB else os.getcwd()
LOG_DIR = os.path.join(BASE, "explorer_logs")
CACHE_DIR = os.path.join(BASE, "explorer_cache")
print("Running in Colab" if IN_COLAB else f"Running outside Colab, in {BASE}")

## 2. Check the GPU

In [ ]:
import subprocess, torch
if not torch.cuda.is_available():
    print("No GPU found. In Colab: Runtime -> Change runtime type -> T4 GPU, then run the notebook again.")
    print("(It will still try to run on the CPU, but a single run would take many minutes.)")
else:
    p = torch.cuda.get_device_properties(0)
    gb = p.total_memory / 2**30
    print(f"GPU: {p.name}, {gb:.0f} GB, compute capability {p.major}.{p.minor}")
    print("bfloat16:", "native" if p.major >= 8 else "emulated (works, slower)")
    print("Plan:", "model and transcoders on the GPU" if gb >= 20 else
          "model on the GPU (image encoder parked in CPU memory), transcoders in CPU memory (fits a T4)")
    print(subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

## 3. Install

In [ ]:
%pip install -q "transformers>=4.56,<6" accelerate safetensors huggingface_hub fastapi "uvicorn>=0.23"

## 4. Get the app files
Uses an `explorer/` folder next to the notebook if there is one (for example, one you uploaded to `/content` with the Files panel). Otherwise it clones `REPO_URL`.

In [ ]:
import os, sys, subprocess

def find_local():
    here = [os.getcwd(), "/content", os.path.join(os.getcwd(), REPO_SUBDIR), os.path.join("/content", REPO_SUBDIR)]
    for d in here:
        if os.path.exists(os.path.join(d, "explorer", "app.py")):
            return os.path.abspath(d)

ROOT = find_local()
if ROOT:
    print("Using the explorer/ folder in", ROOT)
elif "YOUR-USER" in REPO_URL:
    raise SystemExit("REPO_URL is still the placeholder. Set it in the Settings cell, or upload the explorer/ folder "
                     "(Files panel on the left) to /content and run this cell again.")
else:
    dest = "/content/gemma-explorer-src" if IN_COLAB else os.path.abspath("gemma-explorer-src")
    if os.path.exists(os.path.join(dest, ".git")):
        subprocess.run(["git", "-C", dest, "pull", "-q"], check=False)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, dest], check=True)
    ROOT = os.path.join(dest, REPO_SUBDIR)
    print("Cloned", REPO_URL, "->", ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
print(sorted(os.listdir(os.path.join(ROOT, "explorer"))))

## 5. Hugging Face token
Reads the Colab secret `HF_TOKEN` if you added one (key icon on the left, with *Notebook access* on). Otherwise it asks in a hidden prompt. The token is kept only in this session's memory and is never printed.

In [ ]:
import os, getpass
_t = None
if IN_COLAB:
    try:
        from google.colab import userdata
        _t = userdata.get("HF_TOKEN")
    except Exception:
        _t = None      # no secret, or notebook access not granted
_t = _t or os.environ.get("HF_TOKEN") or getpass.getpass("Hugging Face token (input is hidden): ")
os.environ["HF_TOKEN"] = _t.strip()
del _t

from huggingface_hub import hf_hub_download
try:
    from huggingface_hub.errors import GatedRepoError, HfHubHTTPError
except ImportError:
    from huggingface_hub.utils import GatedRepoError, HfHubHTTPError
try:
    hf_hub_download("google/gemma-3-4b-it", "config.json", token=os.environ["HF_TOKEN"])
    print("Token works and the Gemma licence is accepted.")
except GatedRepoError:
    raise SystemExit("Your token works, but this account hasn't accepted the Gemma licence yet. Open "
                     "https://huggingface.co/google/gemma-3-4b-it, accept it, then run this cell again.")
except HfHubHTTPError as e:
    raise SystemExit("Hugging Face refused the token (" + type(e).__name__ + "). Check the token and run this cell again.")

## 6. Where to save runs and notes (optional Google Drive)
Every run and every note is saved as JSON. With `SAVE_TO_DRIVE` ticked in Settings, they go to your Drive and survive the session.

In [ ]:
if SAVE_TO_DRIVE:
    if not IN_COLAB:
        raise SystemExit("Google Drive mounting only works in Colab; untick SAVE_TO_DRIVE.")
    from google.colab import drive
    drive.mount("/content/drive")
    LOG_DIR = f"/content/drive/MyDrive/{DRIVE_FOLDER}/logs"
    CACHE_DIR = f"/content/drive/MyDrive/{DRIVE_FOLDER}/cache"     # the fitted directions are reused next time
os.makedirs(LOG_DIR, exist_ok=True); os.makedirs(CACHE_DIR, exist_ok=True)
print("Runs and notes ->", LOG_DIR)
print("Cache          ->", CACHE_DIR)

## 7. Load Gemma 3 4B-it
Downloads the model (about 8.6 GB) and the layer-17 sparse autoencoder, then loads them onto the GPU. This takes 2–5 minutes the first time.

In [ ]:
import torch
from explorer import app as ex
big_gpu = torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory > 20 * 2**30
ex.init(log_dir=LOG_DIR, cache_dir=CACHE_DIR, dtype=DTYPE, tc_device="cuda" if big_gpu else "cpu")

## 8. Transcoders and directions
**Transcoders** (Gemma Scope 2, one per layer, 16,384 features each) turn the model's internal state into readable features. Only their encoder halves are downloaded.

**Directions fitted here** (well under a minute; cached afterwards):
- the **persona** axis at every layer: role-play characters vs the plain assistant on the same everyday messages. It uses a small built-in set of 72 prompts written for this notebook (see `CHARACTERS` and `MESSAGES` in `explorer/app.py`). The showcase prompts are left out of it.
- the 0 point of the **refusal probe** at layer 14. The direction itself comes from the project. It is the mean difference between harmful and harmless requests, and its size sets 1. The 0 point is the plain assistant on everyday messages.
- any groups you gave in `MY_CONCEPT_GROUPS` (section 10).

The printed "separation" is how often a concept prompt scores above a control prompt (AUROC; 1.00 = perfect). It is measured on the same prompts, so it is optimistic.

In [ ]:
if PRELOAD_TRANSCODERS:
    ex.preload_transcoders()
else:
    print("Transcoders will be fetched layer by layer when first needed (the first run fetches the watch-list layers).")
ex.fit_concepts(groups_file=MY_CONCEPT_GROUPS or None)
print(ex.gpu_memory())

## 9. Open the explorer
This starts the web app inside the notebook and shows it below. **Use the link to open it in its own browser tab**, which gives it much more room. The app keeps running while this runtime is connected, and you can still run the cells further down at the same time.

New to it? Open *New here? What this shows* at the top of the page. Then load a preset from the *Showcase pairs* group and press **Run**. Run its control too, and put the two side by side in **Compare**.

In [ ]:
url = ex.serve(PORT)
if IN_COLAB:
    from google.colab import output
    try:
        output.serve_kernel_port_as_window(PORT, anchor_text="Open the explorer in a new tab")
    except TypeError:
        output.serve_kernel_port_as_window(PORT)
    output.serve_kernel_port_as_iframe(PORT, height=900)
else:
    print("Open", url)

## 10. Prefer code? The same analysis from Python
Everything the page shows is also available as plain Python functions. Runs you make here show up in the page's *Runs* list too (refresh it).

**The refusal showcase pair**: a harmful request and a harmless one about the same dog. The refusal probe is read at the last prompt token, where the reply is decided.

In [ ]:
P = {p["name"]: p for p in ex.PRESETS}
case = ex.run_prompt(P["Refusal: a harmful request (poison the neighbour's dog)"]["messages"], gen_tokens=60, label="showcase: refusal")
ctrl = ex.run_prompt(P["Refusal control: a harmless request about the same dog"]["messages"], gen_tokens=60, label="showcase: refusal control")
for r in (case, ctrl):
    last = r["n_prompt"] - 1
    probes = ", ".join(f"{k} {v[last]:.2f}" for k, v in r["probes"].items())
    print(f"[{r['id']}] {r['messages'][-1]['content']}\n  probes at the last prompt token: {probes}\n  reply: {r['reply'][:300]!r}\n")

**Which features are active where the reply is decided?** These are the strongest labelled transcoder features at every third layer, at the last prompt token. `atlas` and `v16` are the project's two label sets.

In [ ]:
def show_features(run, layers=range(0, 34, 3), k=3):
    print(f"Run {run['id']}: top labelled features at the last prompt token")
    for row in ex.top_features(run["id"], layers=layers, k=k, labelled_only=True):
        print(f"  layer {row['layer']:2d}  {row['key']:>10s}  act {row['act']:7.1f}  [{row['source']:5s}] {row['label']}  ({row['group']})")

show_features(case)

**The persona axis across layers.** This compares Sal, the grumpy diner owner, with the helpful assistant on the same question, at the last prompt token. 0 means "like the plain assistant" and 1 means "like a role-play character". On real weights the two curves are already apart at layer 0 and stay flat: at the last prompt token this axis mostly detects that a role-play system prompt is present, rather than building up a persona layer by layer. The thin lines are the project's *char* axis (tone removed), which exists at fewer layers.

In [ ]:
import matplotlib.pyplot as plt
sal = ex.run_prompt(P["Persona: Sal, the grumpy diner owner"]["messages"], gen_tokens=40, label="showcase: persona")
asst = ex.run_prompt(P["Persona control: a helpful assistant, same question"]["messages"], gen_tokens=40, label="showcase: persona control")
ts, ta = ex.concept_trace(sal["id"]), ex.concept_trace(asst["id"])
fig, ax = plt.subplots(figsize=(7, 3.4))
if "persona" in ts:
    ax.plot(ts["persona"], label="Sal (persona axis)"); ax.plot(ta["persona"], "--", label="assistant (persona axis)")
for t, name, st in ((ts, "Sal", "o-"), (ta, "assistant", "o--")):
    ax.plot(list(t["char"].keys()), list(t["char"].values()), st, lw=0.8, ms=3, label=f"{name} (char axis)")
ax.axhline(0, color="grey", lw=0.5); ax.axhline(1, color="grey", lw=0.5)
ax.set_xlabel("layer"); ax.set_ylabel("0 = assistant, 1 = character"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()
print("Sal:      ", sal["reply"][:200].replace("\n", " "))
print("Assistant:", asst["reply"][:200].replace("\n", " "))

**Steering.** The project found that subtracting the tone-free *character* direction at layer 12 brings back crisis help in character (4% → 40% of replies on held-out characters) while keeping the character's voice. Try it on Bram, a kind innkeeper whose user steps out of the story with a real emergency:

In [ ]:
bram = P["Bram (kind): OOC chest pain"]["messages"]
base = ex.run_prompt(bram, gen_tokens=80, label="Bram OOC chest pain")
steered = ex.run_prompt(bram, gen_tokens=80, steer=[{"id": "char", "layer": 12, "strength": -2}], label="Bram OOC chest pain, char -2 @ L12")
print("Without steering:\n ", base["reply"][:400], "\n")
print("Character direction -2 at layer 12:\n ", steered["reply"][:400])
print("\nSteering ids:", [s["id"] for s in ex.meta()["steer_options"]])

**Fit your own direction (optional).** Write two groups of conversations, a *concept* group and a *control* group, and the notebook fits a direction at every layer. It then appears as an axis on the page (after a reload) and as a `concept:<name>` steering option. The example below fits a harmless "asks for a story" direction. To get the project's other per-layer directions (refusal, a user in danger), build groups from a public dataset of your choice, name them `refusal` / `danger`, save them in the same format, and set `MY_CONCEPT_GROUPS` in Settings to that file.

In [ ]:
import json
topics = ["a lighthouse", "a lost cat", "the first snow", "a broken clock", "a dragon who hates gold", "a quiet library",
         "a train at midnight", "a garden in spring"]
groups = {"story": {
    "concept": [[{"role": "user", "content": f"Write me a short story about {t}."}] for t in topics],
    "control": [[{"role": "user", "content": f"Give me three facts about {t.replace('a ', '', 1)}."}] for t in topics]}}
path = os.path.join(CACHE_DIR, "my_groups_example.json")
json.dump({"groups": groups}, open(path, "w"), indent=1)
ex.fit_concepts(groups_file=path)        # refits persona too; reload the page to see the new 'story' axis
print("Directions now:", list(ex.CONC["dirs"]))

Other useful functions:
- `ex.cell(run_id, layer, pos)`: everything the Cell panel shows.
- `ex.attention(run_id, layer, pos)`: where attention looks from one token.
- `ex.feature("L6.4554")`: the label and examples for one feature.
- `ex.list_runs()` and `ex.load_run(run_id)`: saved runs.
- `ex.add_note("text", run_id)` and `ex.list_notes()`: notes.

`explorer/API.md` lists them all.

In [ ]:
ex.add_note("From the notebook: compared Bram with and without the character direction at layer 12.", steered["id"])
print(len(ex.list_runs()), "runs and", len(ex.list_notes()), "notes saved in", ex.DATA)

## 11. Share what you found
Your runs (with every number the page showed) and your notes are plain JSON files. This cell zips them so you can download and share them. Anyone can load the run files in Python with `json.load`.

In [ ]:
import shutil, time
zip_path = shutil.make_archive(os.path.join(BASE, "explorer_logs_" + time.strftime("%Y%m%d-%H%M")), "zip", LOG_DIR)
print("Zipped", sum(len(f) for _, _, f in os.walk(LOG_DIR)), "files ->", zip_path, f"({os.path.getsize(zip_path) / 1e6:.1f} MB)")
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)

## Troubleshooting
- **"CUDA out of memory"**: use shorter replies and conversations, because each run keeps every layer's state for every token. Then *Runtime → Restart session* and run again. On a T4 the transcoders sit in CPU memory, so they don't add to GPU memory.
- **The page says it cannot reach the backend**: the runtime disconnected, or the server cell was not run. Re-run section 9 (and the cells above it if the runtime restarted).
- **Gated repo / 401 / 403**: accept the licence on the [model page](https://huggingface.co/google/gemma-3-4b-it) with the same account as the token.
- **Very slow on a T4**: that is the bfloat16 emulation. An L4 runtime is several times faster.
- **Feature examples don't load**: they are fetched live from Hugging Face (`mwhanna/gemma-scope-2-4b-it`) and Neuronpedia. Check that the runtime has internet.
- **A run warns about inf/NaN**: you chose float16 in Settings. Switch back to bfloat16.

## Caveats
- Labels are LLM-written best guesses from example texts and the project's prompts, not proofs. The `atlas` set has notes and confidence ratings. The `v16` set keeps only high- and medium-confidence labels.
- The *char*, *tone* and *assistant* axes, the crisis probe and the refusal direction were fit on the project's own prompts (role-play characters, XSTest-style requests, crisis messages). On very different text they may mean less. The *persona* axis is fit here on 72 short, everyday prompts, so it is a rough version of the project's.
- The *char*, *tone* and *assistant* axes were calibrated on reply tokens. *persona* and the refusal probe were calibrated at the last prompt token. Single tokens are noisy, so read averages.
- The steering results quoted in the labels come from the project's tests on its own prompts. Most other strengths and layers are untested.